# GRPO Training for Hierarchical Chain-of-Thought

Fine-tune the SFT-trained HCoT model using Group Relative Policy Optimization (GRPO).

**Reward functions** (defined in `trainer/rewards.py`):
1. **Correctness** — is the `\boxed{}` answer correct? (weight 2.0)
2. **Syntax** — are `[THOUGHT]`/`[SOLUTION]`/`[RETURN]` blocks well-formed? (weight 1.0)
3. **Leaf length** — penalise overly long leaf thoughts (weight 0.5)
4. **Tree depth** — penalise excessive nesting (weight 0.5)
5. **Compression ratio** — reward good solution-to-thought ratio (weight 0.5)
6. **Format** — check `<think>`, `</think>`, `\boxed{}`, and block presence (weight 0.5)

## Setup

In [1]:
import sys, os

# When running in Colab, clone the repo and add lib/ to the path
if "google.colab" in sys.modules:
    if not os.path.exists("cognitive-compression"):
        !git clone https://github.com/anujjamwal/cognitive-compression.git

    !cd cognitive-compression && git pull
    sys.path.insert(0, "cognitive-compression/lib")
    !pip install trl peft math-verify wandb
else:
    # Local: notebook lives inside lib/ already
    sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
    !pip install trl peft math-verify wandb

Cloning into 'cognitive-compression'...
remote: Enumerating objects: 324, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 324 (delta 104), reused 103 (delta 80), pack-reused 176 (from 1)
Receiving objects: 100% (324/324), 604.82 KiB | 5.87 MiB/s, done.
Resolving deltas: 100% (171/171), done.
Already up to date.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 9.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.1/209.1 kB 31.5 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import GRPOTrainer, GRPOConfig

from trainer.rewards import (
    correctness_reward,
    syntax_reward,
    leaf_length_reward,
    depth_reward,
    compression_reward,
    format_reward,
)
from custom_generate import generate

# W&B for experiment tracking
import wandb

In [ ]:
from huggingface_hub import login as notebook_login
notebook_login()

wandb.login()
wandb.init(
    project="hcot-grpo-colab",
    config={
        "model": "OpenMath-Nemotron-1.5B-PruneAware",
        "method": "GRPO",
        "num_generations": 4,
        "reward_weights": [2.0, 1.0, 0.5, 0.5, 0.5, 0.5],
        "num_epochs": 1,
        "batch_size": 2,
        "grad_accum_steps": 4,
        "learning_rate": 1e-5,
        "max_completion_length": 4096,
    },
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:w

## Model & Dataset

In [17]:
MODEL_NAME = "anujjamwal/OpenMath-Nemotron-1.5B-PruneAware"
GRPO_REPO_ID = "anujjamwal/OpenMath-Nemotron-1.5B-PruneAware-grpo"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side='left')

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [5]:
DATASET_NAME = "davidanugraha/OpenMathReasoning-Sampled"
SYSTEM_PROMPT = "Solve the following math problem. Make sure to put the answer (and only answer) inside \\boxed{}."

raw_dataset = raw_dataset = load_dataset(
    DATASET_NAME,
    split="train",
).skip(5000).take(5)
print(f"Dataset size: {len(raw_dataset)}")
print(f"Columns: {raw_dataset.column_names}")

README.md:   0%|          | 0.00/539 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/141M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/149M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/179M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/92544 [00:00<?, ? examples/s]

Dataset size: 5
Columns: ['id', 'question', 'expected_answer', 'problem_source', 'generated_solution', 'pass_rate_72b_tir', 'used_in_kaggle']


In [6]:
def to_grpo_format(example):
    """Convert dataset example to GRPO conversational format.

    Returns a `prompt` (list of message dicts for system + user) and
    keeps `expected_answer` so it is passed as a kwarg to reward functions.
    """
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["question"]},
        ],
        "expected_answer": example["expected_answer"],
    }


dataset = raw_dataset.map(
    to_grpo_format,
    remove_columns=[c for c in raw_dataset.column_names if c != "expected_answer"],
)
print(f"GRPO dataset size: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
print(f"Sample prompt: {dataset[0]['prompt']}")

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

GRPO dataset size: 5
Columns: ['expected_answer', 'prompt']
Sample prompt: [{'content': 'Solve the following math problem. Make sure to put the answer (and only answer) inside \\boxed{}.', 'role': 'system'}, {'content': 'Evaluate the integral \\(\\int \\frac{1}{x^2 - x + 1} \\, dx\\).', 'role': 'user'}]


## Reward Functions

All six reward functions are defined in `trainer/rewards.py`. Quick sanity check below.

In [7]:
# --- Sanity check reward functions on synthetic examples ---

good_completion = [
    {"role": "assistant", "content": (
        "<think>\n"
        "[THOUGHT] We need to solve 2+2.\n"
        "[THOUGHT] 2+2 = 4. [SOLUTION] The sum is 4. [RETURN]\n"
        "[SOLUTION] The answer is 4. [RETURN]\n"
        "</think>\n"
        "\\boxed{4}"
    )}
]

bad_syntax = [
    {"role": "assistant", "content": (
        "<think>\n"
        "[THOUGHT] unclosed block\n"
        "</think>\n"
        "\\boxed{4}"
    )}
]

no_format = [
    {"role": "assistant", "content": "The answer is 4."}
]

test_completions = [good_completion, bad_syntax, no_format]
test_answers = ["4", "4", "4"]

print("Correctness :", correctness_reward(test_completions, expected_answer=test_answers))
print("Syntax      :", syntax_reward(test_completions))
print("Leaf length :", leaf_length_reward(test_completions))
print("Depth       :", depth_reward(test_completions))
print("Compression :", compression_reward(test_completions))
print("Format      :", format_reward(test_completions))

Correctness : [1.0, 1.0, 0.0]
Syntax      : [1.0, 0.0, 1.0]
Leaf length : [1.0, 0.0, 0.0]
Depth       : [1.0, 0.0, 1.0]
Compression : [1.0, 0.0, 0.0]
Format      : [1.0, 0.6667, 0.0]


## PEFT / LoRA (optional)

Set `USE_LORA = True` to train with LoRA instead of full fine-tuning. This reduces trainable parameters from ~1.5B to ~10M, dramatically cutting optimizer memory and enabling cheaper GPUs.

In [8]:
USE_LORA = False

peft_config = None
if USE_LORA:
    from peft import LoraConfig
    peft_config = LoraConfig(
        r=32,
        lora_alpha=64,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        lora_dropout=0.05,
        task_type="CAUSAL_LM",
    )
    print(f"LoRA enabled: ~{sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.1f}M trainable params")
else:
    print("Full fine-tuning enabled")

Full fine-tuning enabled


## Training

In [9]:
# TensorBoard (optional — W&B is also enabled for remote monitoring)
%load_ext tensorboard
%tensorboard --logdir ./hcot-nemotron-1.5b-grpo

<IPython.core.display.Javascript object>

In [ ]:
training_args = GRPOConfig(
    output_dir="./hcot-nemotron-1.5b-grpo",
    hub_model_id=GRPO_REPO_ID,

    # Generation
    num_generations=4,
    max_completion_length=4096,
    generation_kwargs={
        "processing_class": None,
        "return_unpruned_output": True,
    },

    # Optimisation
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    beta=0.1,
    bf16=True,
    gradient_checkpointing=True,
    warmup_ratio=0.1,
    weight_decay=0.01,
    lr_scheduler_type="cosine",

    # Reward weighting: correctness >> syntax >> structural rewards
    reward_weights=[2.0, 1.0, 0.5, 0.5, 0.5, 0.5],

    # Logging & checkpointing
    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    report_to=["tensorboard", "wandb"],
    push_to_hub=False,
    mask_truncated_completions=True,
    torch_compile=True,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [11]:
from transformers import TrainerCallback


class WandbSampleCallback(TrainerCallback):
    """Log sample completions and per-reward breakdowns to W&B every N steps."""

    def __init__(self, tokenizer, reward_funcs, dataset, every_n_steps=50, num_samples=3):
        self.tokenizer = tokenizer
        self.reward_funcs = reward_funcs
        self.reward_names = [
            "correctness", "syntax", "leaf_length",
            "depth", "compression", "format",
        ]
        self.dataset = dataset
        self.every_n_steps = every_n_steps
        self.num_samples = num_samples

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step % self.every_n_steps != 0 or model is None:
            return

        model.eval()
        samples = self.dataset.select(range(min(self.num_samples, len(self.dataset))))
        table = wandb.Table(columns=[
            "step", "prompt", "completion", "expected_answer",
            *self.reward_names, "total_reward",
        ])

        for i, sample in enumerate(samples):
            inputs = self.tokenizer.apply_chat_template(
                sample["prompt"],
                add_generation_prompt=True,
                return_tensors="pt",
                return_dict=True,
            ).to(model.device)

            with torch.no_grad():
                gen = model.generate(**inputs, max_new_tokens=2048, do_sample=True, temperature=0.7)

            completion_text = self.tokenizer.decode(
                gen[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=False
            )
            completion_msg = [{"role": "assistant", "content": completion_text}]

            # Score with each reward function
            scores = {}
            for name, fn in zip(self.reward_names, self.reward_funcs):
                if name == "correctness":
                    r = fn([completion_msg], expected_answer=[sample["expected_answer"]])
                else:
                    r = fn([completion_msg])
                scores[name] = r[0]

            weights = [2.0, 1.0, 0.5, 0.5, 0.5, 0.5]
            total = sum(w * scores[n] for w, n in zip(weights, self.reward_names))
            prompt_text = sample["prompt"][-1]["content"][:200]

            table.add_data(
                state.global_step, prompt_text, completion_text[:1000],
                sample["expected_answer"],
                *[scores[n] for n in self.reward_names], total,
            )

        wandb.log({"sample_completions": table}, step=state.global_step)
        model.train()

In [18]:
reward_funcs = [
    correctness_reward,
    syntax_reward,
    leaf_length_reward,
    depth_reward,
    compression_reward,
    format_reward,
]

sample_callback = WandbSampleCallback(
    tokenizer=tokenizer,
    reward_funcs=reward_funcs,
    dataset=dataset,
    every_n_steps=50,
    num_samples=3,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_funcs,
    args=training_args,
    train_dataset=dataset,
    callbacks=[sample_callback],
    peft_config=peft_config,
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [19]:
torch.cuda.empty_cache()
trainer.train()

InductorError: RuntimeError: PyTorch is checking whether allow_tf32_new is enabled for cuBlas matmul,Current status indicate that you have used mix of the legacy and new APIs to set the TF32 status for cublas matmul. We suggest only using the new API to set the TF32 flag. See also: https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices

Set TORCHDYNAMO_VERBOSE=1 for the internal stack trace (please do this especially if you're reporting a bug to PyTorch). For even more developer context, set TORCH_LOGS="+dynamo"


In [ ]:
merged_model = trainer.model.merge_and_unload()
merged_model.push_to_hub(GRPO_REPO_ID)
tokenizer.push_to_hub(GRPO_REPO_ID)
wandb.finish()

## Test Generation

In [ ]:
from custom_generate import generate

model.eval()

sample_prompt = dataset[9000]["prompt"]
inputs = tokenizer.apply_chat_template(
    sample_prompt,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

gen_out = model.generate(
    **inputs,
    max_new_tokens=4096,
    prune_aware=True,
    processing_class=tokenizer,
    custom_generate=generate._sample,
)

output = tokenizer.decode(gen_out[0], skip_special_tokens=False)
print(output.replace("\\n", "\n"))
print(f"\nOutput tokens: {gen_out.shape[-1]}")
print(f"Expected answer: {dataset[10]['expected_answer']}")